In [6]:
# === Cell 0: Env & (optional) installs ===
import os
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"   # avoid torchvision import issues

# If you need to install/repair deps INSIDE the notebook, un-comment:
# %pip install --quiet --no-cache-dir "transformers==4.44.2" "accelerate==0.34.0" "peft==0.12.0" "datasets>=2.20.0" bitsandbytes sentencepiece

import torch
print("CUDA:", torch.cuda.is_available(), "Device count:", torch.cuda.device_count())

CUDA: True Device count: 1


In [7]:
# === Cell 1: Paths & Base Model ===
from pathlib import Path
import time

BASE_MODEL = "mistralai/Mistral-7B-v0.1"   # change if you prefer another 7B model 
RUN_NAME   = f"mistral7b-lora-{time.strftime('%Y%m%d-%H%M')}"

DATA_DIR   = Path("/workspace/data/processed")
TRAIN_PATH = DATA_DIR / "train.jsonl"        # put your big dataset here (Alpaca-style)
VAL_PATH   = DATA_DIR / "val.jsonl"          # optional; if missing we'll auto-split

OUT_DIR    = Path(f"/workspace/adapters/{RUN_NAME}")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_LEN    = 1024   # context length (adjust based on VRAM)

In [8]:
# === Cell 2: Load dataset (Alpaca-style) ===
from datasets import load_dataset, DatasetDict
import json, random

if not TRAIN_PATH.exists():
    # Fallback tiny sample if you forgot to place data
    print("WARNING: train.jsonl not found; creating a tiny sample.")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    sample = [
        {"instruction":"Translate to French", "input":"Hello", "output":"Bonjour"},
        {"instruction":"Summarize", "input":"Large models are useful for...", "output":"They help with many NLP tasks."},
        {"instruction":"What is 2+3?", "input":"", "output":"5"},
        {"instruction":"Author of Hamlet?", "input":"", "output":"William Shakespeare"},
    ]
    with open(TRAIN_PATH, "w", encoding="utf-8") as f:
        for r in sample:
            f.write(json.dumps(r, ensure_ascii=False)+"\n")

# Load as train split
train = load_dataset("json", data_files=str(TRAIN_PATH))["train"]

# If no val file, create split
if VAL_PATH.exists():
    val = load_dataset("json", data_files=str(VAL_PATH))["train"]
else:
    # 95/5 split
    split = train.train_test_split(test_size=0.05, seed=42)
    train, val = split["train"], split["test"]

ds = DatasetDict({"train": train, "validation": val})
ds

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 3
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 1
    })
})

In [9]:
# === Cell 3: Format to prompts & labels ===
def to_prompt(ex):
    instr = ex.get("instruction", "")
    inp   = ex.get("input", "")
    out   = ex.get("output", "")
    prompt = f"### Instruction:\n{instr}\n\n### Input:\n{inp}\n\n### Response:\n"
    return {"prompt": prompt, "text": prompt + out, "labels": out}

ds = ds.map(to_prompt)
ds["train"][0]

{'instruction': 'What is 2+2?',
 'input': '',
 'output': '4',
 'prompt': '### Instruction:\nWhat is 2+2?\n\n### Input:\n\n\n### Response:\n',
 'text': '### Instruction:\nWhat is 2+2?\n\n### Input:\n\n\n### Response:\n4',
 'labels': '4'}

In [10]:
# === Cell 4: Tokenizer & masking (loss only on the response) ===
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def tok_fn(ex):
    enc = tok(ex["text"], truncation=True, max_length=MAX_LEN)
    # mask prompt tokens in labels
    prompt_ids = tok(ex["prompt"], truncation=True, max_length=MAX_LEN)["input_ids"]
    labels = enc["input_ids"][:]
    m = min(len(prompt_ids), len(labels))
    labels[:m] = [-100] * m
    enc["labels"] = labels
    return enc

cols_to_remove = [c for c in ds["train"].column_names if c not in ("text","prompt","labels")]
ds_tok = ds.map(tok_fn, remove_columns=cols_to_remove, num_proc=None)
ds_tok

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/mistralai/Mistral-7B-v0.1.
401 Client Error. (Request ID: Root=1-68d79f96-7183f4fa3c0c71db23b39d35;2735122d-5965-4821-a580-d09d46c1a44c)

Cannot access gated repo for url https://huggingface.co/mistralai/Mistral-7B-v0.1/resolve/main/config.json.
Access to model mistralai/Mistral-7B-v0.1 is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
# === Cell 5: 4-bit loading + LoRA config ===
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from transformers import DataCollatorForLanguageModeling

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_cfg,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

# Prepare for k-bit training (handles layernorm & embeddings casting)
from transformers import AutoConfig
from peft.utils.other import prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)

# LoRA setup — commonly used target modules for Mistral/LLaMA
lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=32,                 # increase to 64 if VRAM allows (better capacity)
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

dcoll = DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)

In [ ]:
# === Cell 6: TrainingArguments & Trainer ===
from transformers import TrainingArguments, Trainer

EPOCHS = 3
args = TrainingArguments(
    output_dir=str(OUT_DIR),
    per_device_train_batch_size=1,       # keep at 1 for 7B in 4-bit on 16GB
    gradient_accumulation_steps=16,      # effective batch size = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    num_train_epochs=EPOCHS,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    evaluation_strategy="steps",
    eval_steps=500,
    fp16=False,                          # bfloat16 works under the hood with 4-bit; leave fp16 False
    bf16=torch.cuda.is_available(),      # use bf16 on Ampere+ GPUs
    report_to="none",                    # or "wandb" if you configured it
    gradient_checkpointing=True,         # important for memory
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    data_collator=dcoll,
)

trainer.train()

In [ ]:
# === Cell 7: Save LoRA adapter + tokenizer ===
trainer.model.save_pretrained(str(OUT_DIR))
tok.save_pretrained(str(OUT_DIR))
print("✅ Saved LoRA adapter to:", OUT_DIR)

In [ ]:
# === Cell 8: Quick generation test (no pipeline) ===
from transformers import GenerationConfig

def gen(prompt: str, max_new_tokens=128):
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    gen_cfg = GenerationConfig(max_new_tokens=max_new_tokens, do_sample=False)
    with torch.no_grad():
        out = model.generate(**inputs, generation_config=gen_cfg)
    return tok.decode(out[0], skip_special_tokens=True)

test_prompt = "### Instruction:\nTranslate to French\n\n### Input:\nHello\n\n### Response:\n"
print(gen(test_prompt, max_new_tokens=32))